# Getting started


This page offers a quick tour of `xsnow` after you have [installed](./installation.md) the package.

## 1. Quickstart

In [13]:
import xsnow

# Option A: load small sample dataset (downloads on first use)
xs = xsnow.single_profile_timeseries()

# Option B: read a local directory or file
#           e.g., download sample data from xsnow:
# datapath = xsnow.sample_data.snp_gridded_dir()
# xs = xsnow.read(f"{datapath}/pros/gridded")

# Inspect basic structure
print(xs.sizes)
print(xs.coords)

# Select a single profile (one time, slope, realization)
profile = xs.isel(location=0, time=-1, slope=0, realization=0)

# Inspect profile data like a spreadsheet
print(profile.to_dataframe())

# Extract density of snow layers
density = profile["density"]

[i] xsnow.xsnow_io: Loading 2 datasets eagerly with 13 workers...
[i] xsnow.utils: Slope coordinate 'inclination' varies by location. Preserving (location, slope) dimensions as allow_per_location=True.
[i] xsnow.utils: Slope coordinate 'azimuth' varies by location. Preserving (location, slope) dimensions as allow_per_location=True.


Frozen({'location': 1, 'time': 381, 'slope': 1, 'realization': 1, 'layer': 59})
Coordinates:
    altitude     (location) float64 8B 1.681e+03
    azimuth      (location, slope) int64 8B 270
    inclination  (location, slope) int64 8B 16
    latitude     (location) float64 8B 47.1
  * location     (location) <U29 116B 'Kasererwinkl_C6'
    longitude    (location) float64 8B 11.62
  * time         (time) datetime64[ns] 3kB 2024-01-17T16:00:00 ... 2024-02-02...
  * slope        (slope) int64 8B 0
  * realization  (realization) int64 8B 0
  * layer        (layer) int64 472B 0 1 2 3 4 5 6 7 ... 51 52 53 54 55 56 57 58
    z            (location, time, slope, realization, layer) float32 90kB -75...
       ColdContentSnow     DW   HN12  HN24  HN3  HN6  HN72_24  HS_meas  \
layer                                                                    
0               -1.453  348.0  2.163  7.97  0.0  0.0     7.97      NaN   
1               -1.453  348.0  2.163  7.97  0.0  0.0     7.97      NaN   
2 

## 2. The `xsnowDataset` data model
Unlike data models that treat each *single snow profile* as a standalone object, `xsnow` organizes data on a labeled grid. That means one dataset can hold:

- multiple locations,
- multiple times,
- multiple slope/aspect configurations per location and time,
- multiple realizations (e.g. different weather forcings, ensemble members), and
- each full stratigraphy (layered snow profile) underneath those.

Formally, the core dimensions are:

1. **`location`**
   Where is this profile? We typically attach geographic metadata as *coordinates* on this dimension (e.g. `latitude`, `longitude`, `easting`, `northing`, `altitude`).

2. **`time`**
   When is this profile valid?
   Note: `time` is *timezone-naive* and reflects whatever the data source used. How to handle time zones correctly is covered in [Handling timezones](./timezones).

3. **`slope`**
   Same physical location and time, but different terrain orientation. xsnow attaches the coordinates
     - `inclination` (degrees from horizontal)
     - `azimuth` (degrees from north; e.g., 180° = south)
     
     If you only have one slope/aspect, the dimension length is 1.
   

4. **`realization`**
   Parallel "versions" of the snowpack for the same `(location, time, slope)` — for example:
   - different weather inputs (multi model, or ensemble model members),
   - different physical parametrizations,
   - observed vs. modeled,
   - different land cover classes, etc.  
   If you don't use this dimension, it's still present but length 1. `.squeeze()` it out if you prefer.
   
5. **`layer`**
   The vertical snow stratigraphy. Variables defined along `layer` describe per-layer properties: density, grain size, temperature, etc. There is also a vertical coordinate variable called `z` (zero at the snow surface, positive upward, negative downward).


Those five dimensions give you a consistent, rectangular grid. On top of them, we store two broad kinds of variables:

| Kind of variable                          | Typical dims                                             |
|------------------------------------------|----------------------------------------------------------|
| **Profile-level / scalar per profile**     | `(location, time, slope, realization)`                   |
| **Layer-level / stratigraphy variable** | `(location, time, slope, realization, layer)`            |

Example:
- `HS` (total snow height) is profile-level and therefore does not have a `layer` dimension.
- `density` is a layer property and therefore it *does* have the `layer` dimension.


**Why this structure?**

- You can run fully vectorized analysis and statistics across large subsets of the data.  
  For example: *“Give me all weak layers with poor stability on north-facing slopes after a certain date”* becomes a straightforward selection and filter on the dataset---no manual loops, no filename parsing, no ad hoc per-station scripts.

- The entire dataset can be serialized to (and restored from) NetCDF efficiently, lazily, and it can be subset on parsing.

- By adopting this data model, `xsnow` can directly leverage the high-performance [`xarray`](https://docs.xarray.dev/en/stable/) ecosystem.  
  This not only gives us convenient labeled indexing, resampling, and computation, but also sets up `xsnow` for future-friendly storage formats like *zarr* / *geozarr*, which are increasingly used for scalable, cloud-native geoscience data.


```{admonition} An xsnowDataset feels like an xarray.Dataset
:class: note

While `xsnow` introduces its own class, `xsnowDataset`, we’ve taken meticulous care to make an `xsnowDataset` *behave and feel* just like an `xarray.Dataset`.

- **Full compatibility:**  
  Any method you can call on an `xarray.Dataset` can also be called on an `xsnowDataset`.

- **Underlying xarray object:**  
  Every `xsnowDataset` contains a regular `xarray.Dataset` holding the actual data, accessible via the attribute `xs.data`.  
  If you want to explicitly export to a pure `xarray.Dataset`, simply call:

      xs.to_xarray()

- **Automatic method forwarding:**  
  Any attribute or method that’s not part of `xsnowDataset` is automatically forwarded to the underlying `xarray.Dataset`.  
  For example:

      xs.info()      # under the hood calls xs.data.info()
      xs['density']  # under the hood calls xs.data['density']

- **Helpful auto-complete organization:**  
  Python’s tab completion works seamlessly and is structured to help you explore:

      xs.       # + TAB: shows xsnow-native methods and coordinates
      xs.data.  # + TAB: shows all xarray.Dataset methods available on xs as well

This design keeps the workflow intuitive: you get all the power of `xarray`, plus the specialized snow science functionality of `xsnow`.
```

## 3. What to explore next

Use this section (or the website table of contents) to jump into the tutorials that match your goal.

- [Reading and writing](./reading_writing): How to load SNOWPACK and other formats, inspect metadata, and write NetCDF in the `xsnow` layout.
- [Subsetting and filtering](./subsetting_filtering): Use `.sel()` and `.isel()` to target locations, times, slopes, and realizations, plus filter by variable conditions.
- [Combining datasets](./combining_data): Concatenate or combine without breaking stratigraphies.
- [Inspecting your data in tables](./tabular_inspection): Convert profiles to pandas tables for quick QA or reporting.
- [Handling timezones](./timezones): Understand timezone assumptions and safe conversions.
- [Grain types and other labeled data](./gtypes_labels): Translate numeric codes to readable labels.
- **Extensions**: Learn how to plug in new science methods or workflows. Start with [A common framework](./extensions/application_scenarios), before checking out the different application scenarios for extending xsnow to customize your workflows.
